# End-to-end BPS question agent

The pipeline lives in `tilik_agent.agent`. This notebook calls `stream_question`, the same entry the API uses, and runs golden question 6.
The SQL role is read-only; configure `.env`, PostgreSQL, and the PGVector collection before invoking the graph.
No web search is used.


In [ ]:
import json

from tilik_agent.agent.answer import display_chart_spec
from tilik_agent.agent.runner import stream_question
from tilik_agent.agent.schemas import State


In [ ]:
QUESTION = (
    "Provinsi mana saja yang memiliki tiga TPT tertinggi pada Agustus 2025, "
    "dan berapa Angka Partisipasi Sekolah kelompok umur 16–18 tahun di "
    "masing-masing provinsi tersebut pada 2025?"
)

result: State = {"question": QUESTION}
for update in stream_question(QUESTION):
    for node_name, payload in update.items():
        print(f"\n=== node: {node_name} ===", flush=True)
        if node_name == "plan_question":
            print(json.dumps({k: payload.get(k) for k in
                              ("source_route", "plan", "plan_error")},
                             ensure_ascii=False, indent=2, default=str))
        elif node_name in ("run_retrieval", "run_sql", "run_both"):
            counts: dict[str, int] = {}
            for row in payload.get("observations", []):
                key = str(row.get("dataset_id"))
                counts[key] = counts.get(key, 0) + 1
            print(json.dumps({
                "step_status": payload.get("step_status"),
                "observations_by_dataset": counts,
                "document_count": len(payload.get("documents", [])),
                "sql_error": payload.get("sql_error"),
                "sql_warnings": payload.get("sql_warnings"),
                "retrieval_error": payload.get("retrieval_error"),
                }, ensure_ascii=False, indent=2, default=str))
        elif node_name == "write_answer":
            print(payload.get("answer"))
        elif node_name == "build_chart":
            print(json.dumps(payload.get("chart"), ensure_ascii=False, indent=2, default=str))
            if payload.get("chart_error"):
                print(payload["chart_error"])
            display_chart_spec(payload.get("chart"))
        elif "answer" in payload:
            print(payload["answer"])
        result.update(payload)

print("\n=== final answer ===")
print(result["answer"])
